<a href="https://colab.research.google.com/github/EsserMishelle/short-term-stock-forecast/blob/main/05_return_based_linear_and_rf_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Return-Based Linear and Random Forest Models
The objective of the return-based models is to evaluate whether predicting daily returns instead of price levels improves forecasting performance. Because returns are typically more stationary than prices, these models test whether return-based features capture short-term market dynamics more effectively.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit

import seaborn as sns
import os
import time

### Load the Clean Dataset from Google Drive Obtained From Yahoo Finance

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Define folder path within the drive
folder_path = '/content/drive/MyDrive/stocks'
os.makedirs(folder_path, exist_ok=True)

# Define the assets folder path
assets_folder_path = '/content/drive/MyDrive/stocks/assets'
os.makedirs(assets_folder_path, exist_ok=True)

Mounted at /content/drive


In [3]:
# --- Load the clean and flatten file ---
all_stocks_ready_2021_2026_path = os.path.join(folder_path, 'all_stocks_ready_2021_2026.csv')

df = pd.read_csv(all_stocks_ready_2021_2026_path,
    index_col='Date',
    parse_dates=True
)
df

,nvda_open,nvda_high,nvda_low,nvda_close,nvda_volume,qqq_open,qqq_high,qqq_low,qqq_close,qqq_volume,...,tsm_volume,amd_open,amd_high,amd_low,amd_close,amd_volume,vix_open,vix_high,vix_low,vix_close
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,13.067502,13.614215,12.926149,13.076726,560640000,305.791557,305.966256,296.155214,300.163086,45305900,...,11262100,92.110001,96.059998,90.919998,92.300003,51802600,23.040001,29.190001,22.559999,26.969999
2021-01-05,13.063014,13.405800,13.050300,13.367159,322760000,299.173289,302.909444,299.173289,302.637695,29323400,...,10583600,92.099998,93.209999,91.410004,92.769997,34208000,26.940001,28.600000,24.799999,25.340000
2021-01-06,13.185421,13.207858,12.550707,12.579126,580424000,297.921395,302.657089,296.931569,298.445435,52809600,...,10609300,91.620003,92.279999,89.459999,90.330002,51911700,25.480000,26.770000,22.139999,25.070000
2021-01-07,12.931133,13.340234,12.850361,13.306578,461480000,301.104442,306.500020,301.075331,305.665466,30394800,...,13556100,91.330002,95.510002,91.199997,95.160004,42897200,23.670000,23.910000,22.250000,22.370001
2021-01-08,13.325028,13.383364,13.005427,13.239518,292528000,307.955647,309.945042,305.762470,309.595673,33955800,...,18976800,95.980003,96.400002,93.269997,94.580002,39816400,22.430000,23.340000,21.420000,21.559999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-29,191.339996,193.479996,186.059998,192.509995,171764400,632.650024,633.669983,618.270020,629.429993,79944000,...,13844800,254.660004,260.529999,240.910004,252.179993,31685200,16.040001,19.740000,16.020000,16.879999
2026-01-30,191.210007,194.490005,189.470001,191.130005,179489500,625.710022,628.260010,619.299988,621.869995,65650700,...,12028100,236.929993,245.240005,234.550003,236.729996,40035700,18.719999,19.270000,16.670000,17.440001
2026-02-02,187.199997,190.300003,184.880005,185.610001,165794100,618.700012,628.489990,618.659973,626.140015,49020300,...,12459100,235.770004,249.970001,235.000000,246.270004,36308100,19.950001,19.959999,16.080000,16.340000


#### The time frame of the dataset is: 2021-01-04 to 2026-02-04.  There are 28 columns, 1278 rows and no null values.

## Return-Based Models
### Evaluates predictive performance using return targets

Return-based model predicts future returns and converts them back to price for comparison using:
future_price = last_price × (1 + predicted_return).

Price levels exhibit strong persistence (e.g., corr(y_30, nvda_price_lag1)= 0.98), meaning a price-based model may rely heavily on autoregressive structure. Modeling returns reduces this persistence and forces the model to learn underlying relationships rather than simply extending recent price levels.

Return modeling may not necessarily lower lower MAE dramatically , but it improves: statistical correctness, stability across regimes, compatibility with nonlinear models.

In [4]:
split_date = pd.to_datetime('2025-02-03') # first business date in Feb, 2025

### Return-Based Feature Engineer

In [5]:
# Add Momentum
df_ret_model= df.copy()

df_ret_model['nvda_ret1'] = df_ret_model['nvda_close'].pct_change(1).shift(1)
df_ret_model['nvda_ret5'] = df_ret_model['nvda_close'].pct_change(5).shift(1)

df_ret_model['nvda_ret_roll5']  = df_ret_model['nvda_ret1'].rolling(5).mean()
df_ret_model['nvda_ret_roll10'] = df_ret_model['nvda_ret1'].rolling(10).mean()

df_ret_model['nvda_ret_vol5'] = (
    df_ret_model['nvda_close'].pct_change()
    .rolling(5)
    .std()
    .shift(1)
)

# Add Volatility
df_ret_model['nvda_ret_vol5']  = df_ret_model['nvda_ret1'].rolling(5).std()
df_ret_model['nvda_ret_vol10'] = df_ret_model['nvda_ret1'].rolling(10).std()

df_ret_model['tsm_ret1'] = df_ret_model['tsm_close'].pct_change().shift(1)
df_ret_model['tsm_ret3']  = df_ret_model['tsm_close'].pct_change(3).shift(1)
df_ret_model['tsm_ret3']  = df_ret_model['tsm_close'].pct_change(3).shift(1)
df_ret_model['tsm_ret10'] = df_ret_model['tsm_close'].pct_change(10).shift(1)
df_ret_model['tsm_ret20'] = df_ret_model['tsm_close'].pct_change(20).shift(1)

df_ret_model['qqq_ret1'] = df_ret_model['qqq_close'].pct_change().shift(1)
df_ret_model['qqq_ret3'] = df_ret_model['qqq_close'].pct_change().shift(3)
df_ret_model['qqq_ret5'] = df_ret_model['qqq_close'].pct_change().shift(5)
df_ret_model['qqq_ret15'] = df_ret_model['qqq_close'].pct_change().shift(15)
df_ret_model['vix_ret1'] = df_ret_model['vix_close'].pct_change().shift(1)
df_ret_model['vix_ret3'] = df_ret_model['vix_close'].pct_change().shift(3)

df_ret_model['amd_ret1'] = df_ret_model['amd_close'].pct_change().shift(1)
df_ret_model['amd_ret3']  = df_ret_model['amd_close'].pct_change(3).shift(1)
df_ret_model['amd_ret5']  = df_ret_model['amd_close'].pct_change(5).shift(1)
df_ret_model['amd_ret10'] = df_ret_model['amd_close'].pct_change(10).shift(1)
df_ret_model['amd_ret20'] = df_ret_model['amd_close'].pct_change(20).shift(1)

df_ret_model['tnx_ret1'] = df_ret_model['tnx_close'].pct_change().shift(1)
df_ret_model['tnx_ret3'] = df_ret_model['tnx_close'].pct_change().shift(3)
# Add Peer Momentum
df_ret_model['tsm_ret_roll5'] = df_ret_model['tsm_ret1'].rolling(5).mean()
df_ret_model['amd_ret_roll5'] = df_ret_model['amd_ret1'].rolling(5).mean()
df_ret_model['qqq_ret_roll5'] = df_ret_model['qqq_ret1'].rolling(5).mean()

df_ret_model['tsm_vol10'] = (
    df_ret_model['tsm_close']
    .pct_change()
    .rolling(10)
    .std()
    .shift(1)
)

df_ret_model['nvda_mom5'] = (
    df_ret_model['nvda_close'].shift(1)
    - df_ret_model['nvda_close'].shift(6)
)

df_ret_model['tsm_mom10'] = (
    df_ret_model['tsm_close'].shift(1)
    - df_ret_model['tsm_close'].shift(11)
)

df_ret_model['t'] = np.arange(len(df_ret_model))

display(df_ret_model.columns)


Index(['nvda_open', 'nvda_high', 'nvda_low', 'nvda_close', 'nvda_volume',
       'qqq_open', 'qqq_high', 'qqq_low', 'qqq_close', 'qqq_volume',
       'tnx_open', 'tnx_high', 'tnx_low', 'tnx_close', 'tsm_open', 'tsm_high',
       'tsm_low', 'tsm_close', 'tsm_volume', 'amd_open', 'amd_high', 'amd_low',
       'amd_close', 'amd_volume', 'vix_open', 'vix_high', 'vix_low',
       'vix_close', 'nvda_ret1', 'nvda_ret5', 'nvda_ret_roll5',
       'nvda_ret_roll10', 'nvda_ret_vol5', 'nvda_ret_vol10', 'tsm_ret1',
       'tsm_ret3', 'tsm_ret10', 'tsm_ret20', 'qqq_ret1', 'qqq_ret3',
       'qqq_ret5', 'qqq_ret15', 'vix_ret1', 'vix_ret3', 'amd_ret1', 'amd_ret3',
       'amd_ret5', 'amd_ret10', 'amd_ret20', 'tnx_ret1', 'tnx_ret3',
       'tsm_ret_roll5', 'amd_ret_roll5', 'qqq_ret_roll5', 'tsm_vol10',
       'nvda_mom5', 'tsm_mom10', 't'],
      dtype='object')

In [6]:
# Define Feature Selection

# Simple model
X_simple_cols = [
    'nvda_ret1',
]

# Expanded model
# Add Rolling Momentume and Volatility
X_exp_cols = [
    'nvda_ret1',
    'nvda_ret_roll5',
    'nvda_ret_vol5',
    'amd_ret3',

    'qqq_ret1',
    'tsm_ret1',

    'amd_ret_roll5',
    'qqq_ret15',
    'vix_ret3',
    'tnx_ret3',
    'tsm_vol10',
]

#### Train, Test, Split and RMSE Evaluate Function

In [7]:
def evaluate_return_model(
    df,
    feature_cols,
    horizon,
    split_date,
    model,
    model_name=None,
    scale=False
):

    df_model = df.copy()

    # Create dynamic target
    df_model['y'] = df_model['nvda_close'].pct_change(horizon).shift(-horizon)

    # Keep only the selected columns
    df_model = df_model[feature_cols + ['y']]

    # Drop NaNs safely
    df_model = df_model.dropna()

    # Split
    train = df_model[df_model.index < split_date]
    test  = df_model[df_model.index >= split_date]

    X_train = train[feature_cols]
    y_train = train['y']

    X_test  = test[feature_cols]
    y_test  = test['y']

    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

    # Fit model
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)

    results = {
        "Horizon": horizon,
        "Train_MAE": mean_absolute_error(y_train, y_pred_train),
        "Test_MAE":  mean_absolute_error(y_test, y_pred_test),
        "Train_RMSE": np.sqrt(mean_squared_error(y_train, y_pred_train)),
        "Test_RMSE":  np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "Train_MAPE": mean_absolute_percentage_error(y_train, y_pred_train),
        "Test_MAPE":  mean_absolute_percentage_error(y_test, y_pred_test)
    }

    return results, y_pred_test, y_test, test.index, model

### Return-Based Simple Linear Regression and Return-Based Random Forest

When restricted to a pure autoregressive specification, the return-based simple feature model performs as well as or better than the expanded feature model across horizons. The additional volatility and cross-asset return features do not yield meaningful improvement in predictive performance. Consequently, the return-based simple model is retained for comparison with other forecasting approaches.

#### Random Forest Hyperparameter Selection
Hyperparameter tuning using time-series cross-validation has been explored. However, tuning does not improve the performance relative to conservative baseline settings. Therefore, a stable set of Random Forest parameters is used for final evaluation.

In [8]:
horizons = [1, 5, 10, 20, 30]

In [9]:
def convert_return_to_price_rmse(
    df_original,
    horizon,
    test_index,
    y_pred_returns,
    split_date
):
    df_price = df_original.copy()

    # Price at time t
    price_t = df_price.loc[test_index, "nvda_close"]

    # True future price
    true_price = df_price["nvda_close"].shift(-horizon).loc[test_index]

    # Predicted future price
    pred_price = price_t * (1 + y_pred_returns)

    rmse_price = np.sqrt(mean_squared_error(true_price, pred_price))

    return rmse_price

In [10]:
def run_return_models_across_horizons(
    df_ret,
    df_price,
    feature_cols,
    horizons,
    split_date,
    model,
    model_name
):

    all_results = []
    all_models_dict = {}

    for h in horizons:

        metrics, y_pred, y_test, test_idx, trained_model = evaluate_return_model(
            df=df_ret,
            feature_cols=feature_cols,
            horizon=h,
            split_date=split_date,
            model=clone(model)
        )

        # Store the trained model for this horizon
        all_models_dict[h] = trained_model

        # Convert to price RMSE
        price_rmse = convert_return_to_price_rmse(
            df_original=df_price,
            horizon=h,
            test_index=test_idx,
            y_pred_returns=y_pred,
            split_date=split_date
        )

        metrics["Model"] = model_name
        metrics["Price_RMSE"] = price_rmse

        all_results.append(metrics)

    return pd.DataFrame(all_results), all_models_dict

In [11]:
from sklearn.base import clone

ret_lr_df, ret_lr_models = run_return_models_across_horizons(
    df_ret=df_ret_model,
    df_price=df,
    feature_cols=X_simple_cols,
    horizons=horizons,
    split_date=split_date,
    model=LinearRegression(),
    model_name="Return_Simple_LR"
)

ret_rf_df, ret_rf_models = run_return_models_across_horizons(
    df_ret=df_ret_model,
    df_price=df,
    feature_cols=X_exp_cols,
    horizons=horizons,
    split_date=split_date,
    model=RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=5,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    ),
    model_name="Return_RF",
)

display(ret_lr_df)
display(ret_rf_df)

ret_lr_df.to_csv(
    os.path.join(folder_path, "return_simple_lr_results.csv"),
    index=False
)

ret_rf_df.to_csv(
    os.path.join(folder_path,"return_rf_results.csv"),
    index=False
)

,Horizon,Train_MAE,Test_MAE,Train_RMSE,Test_RMSE,Train_MAPE,Test_MAPE,Model,Price_RMSE
0,1,0.025058,0.019641,0.033748,0.027804,2.304294e+10,1.160578,Return_Simple_LR,3.889867
1,5,0.056996,0.044334,0.072458,0.058310,1.719543e+00,2.255360,Return_Simple_LR,8.247543
2,10,0.085416,0.058068,0.106544,0.077014,3.827404e+00,3.718927,Return_Simple_LR,11.045315
3,20,0.124910,0.087508,0.157548,0.110238,3.095608e+00,2.796768,Return_Simple_LR,15.381834
4,30,0.159425,0.117638,0.199209,0.142071,3.107480e+00,5.493000,Return_Simple_LR,19.691393


,Horizon,Train_MAE,Test_MAE,Train_RMSE,Test_RMSE,Train_MAPE,Test_MAPE,Model,Price_RMSE
0,1,0.018316,0.019959,0.025030,0.028238,3.870775e+10,1.349344,Return_RF,3.947891
1,5,0.038205,0.045988,0.049017,0.059547,1.254255e+00,2.820025,Return_RF,8.525272
2,10,0.058542,0.057901,0.073851,0.076973,1.947119e+00,2.741947,Return_RF,11.124428
3,20,0.083096,0.092943,0.105971,0.118351,2.916823e+00,3.885447,Return_RF,17.245686
4,30,0.110633,0.124716,0.137670,0.155238,2.278758e+00,5.625254,Return_RF,22.218596


### Price Compare Between Returned-Based and Price-Based Models
MAPE metrics is fine for price models, but it explodes when returns are near 0. MSE is squared dollars, the scale becomes uninintuitive. RMSE penalizes large forecast errors and is in the same unit as the price model so it is directly interpretable.  Model comparisons will be based primarily on out-of-sample RMSE.

### Use Random Forest to find feature importance

In [12]:
h_pick = 30
rf_m = ret_rf_models[h_pick]

feat_imp = (
    pd.DataFrame({"Feature": X_exp_cols, "Importance": rf_m.feature_importances_})
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(feat_imp.head(20))

,Feature,Importance
0,tsm_vol10,0.204214
1,nvda_ret_vol5,0.150714
2,amd_ret_roll5,0.098553
3,nvda_ret_roll5,0.084433
4,tnx_ret3,0.074139
5,qqq_ret15,0.074108
6,amd_ret3,0.070858
7,vix_ret3,0.065764
8,qqq_ret1,0.059692
9,tsm_ret1,0.059141


The return-based Random Forest assigns greater relative importance to short-term volatility measures and cross-asset signals than to NVDA’s own immediate return lag. Semiconductor peers (TSM, AMD), market volatility (VIX), and interest rate movements (TNX) rank prominently among the top predictors. This suggests that, within the nonlinear structure of the model, broader sector and macro conditions contribute comparably, and in some cases more to short-horizon return than its autoregressive momentum alone.

## Conclusion

Across horizons (1, 5, 10, 20, 30 days) linear regression consistently performs slightly better than Random Forest. Errors increase as the forecast horizon grows, which is expected. Both return-based models struggle with longer horizons, reinforcing how difficult short-term return prediction is.